Importação das bibliotecas utilizadas

In [0]:
import requests
import pandas as pd
from datetime import datetime 
from dateutil.relativedelta import relativedelta

Criação do Database "database_analytics", local onde será alocado os datasets

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS database_analytics;

Extração dos dados de IPCA da API do Banco Central do Brasil.

In [0]:
# Criação da Variável com a data de início da série histórica, filtrando os últimos 20 anos.

start_date = ((datetime.today() - relativedelta(years=20)).replace(day=1, month=1)).strftime("%d/%m/%Y")
end_date = datetime.today().strftime("%d/%m/%Y")

response = requests.get(
        f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.10844/dados?formato=json&dataInicial={start_date}&dataFinal={end_date}'
    )

# Validando o status code da API
assert response.status_code == 200, f'Error: {response.status_code}, Response: {response.text}'

ipca_data = response.json()

# Criação do Dataframe Pandas
df_ipca = pd.DataFrame(ipca_data)

Tratamento e visualização da qualidade dos dados

In [0]:
# Ajustando o tipo dos dados
df_ipca["data"] = pd.to_datetime(df_ipca["data"]).dt.date
df_ipca["valor"] = pd.to_numeric(df_ipca["valor"], errors="coerce")

# Verificando se possui valores nulos
df_ipca.isnull().sum()

# Visualizando o dataframe e as estatístcas dos valores
display(df_ipca.head())
df_ipca.describe()


In [0]:
# Convertendo para Spark
df_ipca_spark = spark.createDataFrame(df_ipca)

# Salvando os dados em uma tabela delta na base de dados "database_analytics.dataset_ipca"
df_ipca_spark.write.format("delta").mode("overwrite").saveAsTable("database_analytics.dataset_ipca")
     

Visualização da tabela no Database "database_analytics"

In [0]:
%sql
SELECT * FROM database_analytics.dataset_ipca;